# FFT 关键概念详解

本 Notebook 详细解释以下 FFT 相关概念：
1. `np.fft.fftshift` 的作用
2. 如何根据采样频率计算频域横轴的真实频率值
3. 信号包含直流分量时可能出现的问题及去除方法

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 1. `np.fft.fftshift` 的作用

### 问题背景
FFT 输出的频率排列是：`[0, 1, 2, ..., N/2-1, -N/2, ..., -2, -1]`
这意味着：
- 前半部分：正频率（从 0 到 Nyquist 频率）
- 后半部分：负频率（从 -Nyquist 到 -1）

### fftshift 的功能
将 FFT 结果重新排列，使零频率分量位于中心，频率从负到正排列：
`[-N/2, ..., -1, 0, 1, ..., N/2-1]`

In [ ]:
np.random.seed(42)

N = 8
original_freqs = np.fft.fftfreq(N, 1)
shifted_freqs = np.fft.fftshift(original_freqs)

print("原始 FFT 频率顺序:")
print(original_freqs)
print()
print("使用 fftshift 后的频率顺序:")
print(shifted_freqs)

In [ ]:
sampling_freq = 100
duration = 1
t = np.linspace(0, duration, int(sampling_freq * duration), endpoint=False)

signal = np.sin(2 * np.pi * 10 * t)

fft_result = np.fft.fft(signal)
frequencies = np.fft.fftfreq(len(signal), 1/sampling_freq)

fft_shifted = np.fft.fftshift(fft_result)
freqs_shifted = np.fft.fftshift(frequencies)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(frequencies, np.abs(fft_result), 'b-', marker='o')
axes[0].set_title('FFT 原始输出（未使用 fftshift）')
axes[0].set_xlabel('频率 (Hz)')
axes[0].set_ylabel('幅度')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=0, color='r', linestyle='--', alpha=0.5)

axes[1].plot(freqs_shifted, np.abs(fft_shifted), 'g-', marker='o')
axes[1].set_title('使用 fftshift 后（零频率居中）')
axes[1].set_xlabel('频率 (Hz)')
axes[1].set_ylabel('幅度')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=0, color='r', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### fftshift 总结

**优点：**
- 频谱图更符合人类直觉（零频率在中心）
- 便于观察正负频率的对称性

**注意事项：**
- 使用 `fftshift` 后进行逆变换时，需要先用 `ifftshift` 还原
- 对于实数信号，正负频率幅度对称，通常只需要看正频率部分

## 2. 如何根据采样频率计算频域横轴的真实频率值

### 奈奎斯特采样定理
- 采样频率 $f_s$ 必须大于信号最高频率的 2 倍
- 可观测的最大频率：$f_{Nyquist} = f_s / 2$

### 频率分辨率
- 频率分辨率 $\Delta f = f_s / N$
- 其中 $N$ 是采样点数

### 频率轴计算方法
```python
# 方法1: 使用 np.fft.fftfreq
frequencies = np.fft.fftfreq(N, 1/fs)

# 方法2: 手动计算
frequencies = np.arange(N) * fs / N
frequencies[frequencies > fs/2] -= fs
```

In [ ]:
sampling_freq = 1000  # 采样频率
N = 1000              # 采样点数
duration = N / sampling_freq  # 信号持续时间

print(f"采样频率: {sampling_freq} Hz")
print(f"采样点数: {N}")
print(f"信号时长: {duration} 秒")
print(f"奈奎斯特频率: {sampling_freq/2} Hz")
print(f"频率分辨率: {sampling_freq/N} Hz")
print()

freqs_method1 = np.fft.fftfreq(N, 1/sampling_freq)

freqs_method2 = np.arange(N) * sampling_freq / N
freqs_method2[freqs_method2 > sampling_freq/2] -= sampling_freq

print("方法1 (np.fft.fftfreq) 前10个频率:")
print(freqs_method1[:10])
print()
print("方法2 (手动计算) 前10个频率:")
print(freqs_method2[:10])

In [ ]:
sampling_freq = 1000
duration = 1
t = np.linspace(0, duration, int(sampling_freq * duration), endpoint=False)

freq1 = 50
freq2 = 150
signal = np.sin(2 * np.pi * freq1 * t) + 0.5 * np.sin(2 * np.pi * freq2 * t)

fft_result = np.fft.fft(signal)
frequencies = np.fft.fftfreq(len(signal), 1/sampling_freq)

positive_mask = frequencies >= 0
positive_freqs = frequencies[positive_mask]
positive_magnitude = np.abs(fft_result[positive_mask]) / len(signal) * 2

plt.figure(figsize=(12, 6))
plt.plot(positive_freqs, positive_magnitude, 'b-')
plt.xlabel('频率 (Hz)')
plt.ylabel('幅度')
plt.title(f'正确的频率轴 (fs={sampling_freq} Hz)')
plt.xlim(0, sampling_freq/2)
plt.grid(True, alpha=0.3)

plt.axvline(x=freq1, color='r', linestyle='--', alpha=0.5, label=f'{freq1} Hz')
plt.axvline(x=freq2, color='g', linestyle='--', alpha=0.5, label=f'{freq2} Hz')
plt.legend()

print(f"第一个峰值约在: {positive_freqs[np.argmax(positive_magnitude[:100])]:.1f} Hz")
print(f"第二个峰值约在: {positive_freqs[np.argmax(positive_magnitude[100:200])+100]:.1f} Hz")

plt.show()

### 频率轴计算总结

**关键公式：**
- 第 k 个频点对应的频率：$f_k = k \times f_s / N$
- 当 $k > N/2$ 时，实际频率为 $f_k - f_s$

**常见错误：**
- 忘记除以采样点数 N
- 直接使用索引作为频率值
- 不理解频率分辨率与信号时长的关系

## 3. 信号包含直流分量时可能出现的问题及去除方法

### 什么是直流分量？
直流分量（DC component）是信号的平均值，对应频率为 0 Hz 的分量。

### 直流分量带来的问题：
1. **频谱图畸变**：0 Hz 处出现巨大峰值，掩盖其他频率分量
2. **幅度计算错误**：直流分量不需要乘以 2（只有单边谱需要）
3. **可视化困难**：纵轴范围被直流分量拉大，其他细节看不清

### 去除直流分量的方法：
```python
# 方法1: 减去均值
signal_dc_removed = signal - np.mean(signal)

# 方法2: 在频域去除
fft_result = np.fft.fft(signal)
fft_result[0] = 0  # 将零频率分量置零
signal_dc_removed = np.fft.ifft(fft_result)
```

In [ ]:
sampling_freq = 1000
duration = 1
t = np.linspace(0, duration, int(sampling_freq * duration), endpoint=False)

dc_offset = 5.0
freq1 = 10
signal_with_dc = dc_offset + np.sin(2 * np.pi * freq1 * t)

print(f"信号均值 (直流分量): {np.mean(signal_with_dc):.4f}")
print(f"信号最小值: {np.min(signal_with_dc):.4f}")
print(f"信号最大值: {np.max(signal_with_dc):.4f}")

In [ ]:
fft_with_dc = np.fft.fft(signal_with_dc)
frequencies = np.fft.fftfreq(len(signal_with_dc), 1/sampling_freq)

positive_mask = frequencies >= 0
positive_freqs = frequencies[positive_mask]
magnitude_with_dc = np.abs(fft_with_dc[positive_mask]) / len(signal_with_dc) * 2
magnitude_with_dc[0] /= 2

signal_no_dc = signal_with_dc - np.mean(signal_with_dc)
fft_no_dc = np.fft.fft(signal_no_dc)
magnitude_no_dc = np.abs(fft_no_dc[positive_mask]) / len(signal_no_dc) * 2

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(t, signal_with_dc, 'b-')
axes[0, 0].set_title('含直流分量的时域信号')
axes[0, 0].set_xlabel('时间 (秒)')
axes[0, 0].set_ylabel('幅度')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=dc_offset, color='r', linestyle='--', alpha=0.5, label='直流电平')
axes[0, 0].legend()

axes[0, 1].plot(positive_freqs, magnitude_with_dc, 'b-')
axes[0, 1].set_title('含直流分量的频谱')
axes[0, 1].set_xlabel('频率 (Hz)')
axes[0, 1].set_ylabel('幅度')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xlim(0, 50)

axes[1, 0].plot(t, signal_no_dc, 'g-')
axes[1, 0].set_title('去除直流分量后的时域信号')
axes[1, 0].set_xlabel('时间 (秒)')
axes[1, 0].set_ylabel('幅度')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)

axes[1, 1].plot(positive_freqs, magnitude_no_dc, 'g-')
axes[1, 1].set_title('去除直流分量后的频谱')
axes[1, 1].set_xlabel('频率 (Hz)')
axes[1, 1].set_ylabel('幅度')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xlim(0, 50)

plt.tight_layout()
plt.show()

In [ ]:
print("含直流分量的频谱前5个点:")
for i in range(5):
    print(f"  {positive_freqs[i]:.1f} Hz: 幅度 = {magnitude_with_dc[i]:.4f}")

print()
print("去除直流分量后的频谱前5个点:")
for i in range(5):
    print(f"  {positive_freqs[i]:.1f} Hz: 幅度 = {magnitude_no_dc[i]:.4f}")

### 直流分量处理总结

**识别直流分量：**
- 时域：信号均值不为零
- 频域：0 Hz 处有非零幅度

**去除方法：**
1. **时域去均值**（推荐）：简单高效，`signal - np.mean(signal)`
2. **频域置零**：将 FFT 结果的第一个元素置零
3. **高通滤波**：使用滤波器去除低频分量

**注意事项：**
- 直流分量在双边谱中只出现一次，幅度计算不需要乘以 2
- 去除直流分量后，信号均值为零，更利于观察交流成分
- 某些应用（如电力系统分析）中，直流分量是有用信息，不应去除

## 完整示例：处理含直流分量的真实信号

In [ ]:
sampling_freq = 1000
duration = 2
t = np.linspace(0, duration, int(sampling_freq * duration), endpoint=False)

dc_component = 3.5
freq1 = 5
freq2 = 20
freq3 = 50

signal = (dc_component + 
          np.sin(2 * np.pi * freq1 * t) + 
          0.5 * np.cos(2 * np.pi * freq2 * t) + 
          0.3 * np.sin(2 * np.pi * freq3 * t))

noise = 0.2 * np.random.randn(len(t))
signal_with_noise = signal + noise

print(f"原始信号均值: {np.mean(signal_with_noise):.4f}")

signal_clean = signal_with_noise - np.mean(signal_with_noise)
print(f"去直流后信号均值: {np.mean(signal_clean):.4f}")

fft_result = np.fft.fft(signal_clean)
frequencies = np.fft.fftfreq(len(signal_clean), 1/sampling_freq)

positive_mask = frequencies >= 0
positive_freqs = frequencies[positive_mask]
magnitude = np.abs(fft_result[positive_mask]) / len(signal_clean) * 2

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].plot(t, signal_with_noise, 'r-', alpha=0.5, label='含直流+噪声')
axes[0].plot(t, signal_clean, 'b-', alpha=0.7, label='去直流后')
axes[0].set_xlabel('时间 (秒)')
axes[0].set_ylabel('幅度')
axes[0].set_title('时域信号对比')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(positive_freqs, magnitude, 'g-', linewidth=2)
axes[1].set_xlabel('频率 (Hz)')
axes[1].set_ylabel('幅度')
axes[1].set_title('频谱分析 (去直流后)')
axes[1].set_xlim(0, 100)
axes[1].grid(True, alpha=0.3)

for freq in [freq1, freq2, freq3]:
    axes[1].axvline(x=freq, color='r', linestyle='--', alpha=0.5, label=f'{freq} Hz')
axes[1].legend()

plt.tight_layout()
plt.show()

## 总结

### 1. `np.fft.fftshift`
- 作用：将零频率分量移到频谱中心
- 效果：频谱从 `[0, 1, ..., N/2, -N/2, ..., -1]` 变为 `[-N/2, ..., -1, 0, 1, ..., N/2-1]`
- 用途：便于观察正负频率对称性

### 2. 频率轴计算
- 公式：$f_k = k \times f_s / N$
- 使用 `np.fft.fftfreq(N, 1/fs)` 自动计算
- 频率分辨率 = 采样频率 / 采样点数

### 3. 直流分量问题
- 表现：0 Hz 处有大峰值，掩盖其他频率
- 去除方法：时域去均值（推荐）或频域置零
- 注意：直流分量幅度计算不需要乘以 2